<a href="https://colab.research.google.com/github/RenatGreen-flag/Model-Liniar-Regression/blob/main/goldPricingPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

TROY_OUNCE_TO_GRAM = 31.1035

data_source = {
    "gold": "GC-GOLD.csv",
    "usdidr": "USDIDR-KURS.csv",
    "oil": "CL-CRUDE-OIL.csv",
    "ihsg": "JKSE-IHSG-IDN.csv",
    "treasury-US": "TNX-US-10-YEAR-TREASURE.csv",
    "snp500": "GSPC-S&P_500_AS.csv"
}

LAG_STEPS = [1,3,7]

ROLLING_WINDOWS = [3,7]

df_list = []
for name, path in data_source.items():
    df = pd.read_csv(path, parse_dates=["Date"])

    temp = temp[["Date", "Close"]], rename(columns={"Close"; f"{name}_close"})

    df_list.append(temp.set_index("Date"))

df = pd.concat(df_list, axis=1, join="outer")
df = df.sort_index()

full_range = pd.date_range(start=df.index.min(), end=ddf.index.max(), freq="D")
df = df.reindex(full_range)
df.index.name = "Date"

# missing value handling
price_cols = [col for col in df.columns if "close" in col]
df[price_cols] = df[price_cols].ffill()

df[price_cols] = df[price_cols].bfill()




In [ ]:
# feature engineering gold usd -> idr per gram
df["gold_close_idr_per_oz"] = df["gold_close"] * df[usdidr_close]

df["gold_close_idr_per_gram"] = df["gold_close_idr_per_oz"] / TROY_OUNCE_TO_GRAM

price_cols = price_cols + ["gold_close_idr_per_oz","gold_close_idr_per_gram"]

In [ ]:
# feature engineering : lag feature (H-1, H-3, H-7)

for col in price_cols:
  for lag in LAG_STEPS:
    df[f"{col}_lag{lag}"] = df[col].shift(lag)

In [ ]:
# feature engineering: Rolling statistic per time step
for col in price_cols:
  for window in ROLLING_WINDOWS:
    df[f"{col}_rmean_{window}"] = df[col].rolling(window=window).mean()
    df[f"{col}_rstd_{window}"] = df[col].rolling(window=window).std()
    df[f"{col}_rmin_{window}"] = df[col].rolling(window=window).min()
    df[f"{col}_rmax_{window}"] = df[col].rolling(window=window).max()

In [ ]:
# Featur engineering: persentase perubahan (RETURN HARIAN)
for col in price_cols:
  df[f"{col}_pct_change"] = df[col].pct_change()

df_clean = df.dropna().reset_index()

In [ ]:
# ringkasan & simpan hasil

print("Jumlah baris & kolom setelah preprocessing:", df_clean.shape)
print("\nContoh kolom yang dihasilkan:")
print([c for c in df_clean.columns if "gold_close" in c][:10])
print("\nBaris pertama:")
print(df_clean.head())

df_clean.to_csv("preprocessing_gold_usdidr", index=False)